# Uplift boosting: fmlib / autocampaignxfm parity

Полный `trainer_booster.py` + `evaluater_booster.py` сравнивается с публичным API fmlib для product- и channel-моделей. Оба пайплайна выполняют по три Optuna trials на коротком тестовом grid; сравниваются параметры, метрики, распределения скоров и время.

Из-за объёмных логов causalml полный parity запускается терминальными командами `python -m tools.automl_parity.uplift` для конфигов `parity_uplift_product_*` и `parity_uplift_channel_*`. Ноутбук читает созданные `comparison.json` и сохраняет компактные таблицы результатов.

In [1]:
import json
from pathlib import Path

import polars as pl
from IPython.display import display

FMLIB_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'fmlib').is_dir())
WORKSPACE = FMLIB_ROOT.parent
CONFIG_DIR = FMLIB_ROOT / 'examples/automl/tests/configs'
AUTOCAMPAIGN_ROOT = WORKSPACE / 'autocampaignxfm'
AUTOCAMPAIGN_PYTHON = AUTOCAMPAIGN_ROOT / 'env/bin/python'
assert AUTOCAMPAIGN_PYTHON.is_file()

In [2]:
# Full parity is launched from the terminal to keep verbose causalml/autocampaign logs out of Jupyter I/O.
comparison_paths = {
    scope: WORKSPACE / f'outputs/uplift_parity_{scope}/fmlib/parity/comparison.json'
    for scope in ('product', 'channel')
}
assert all(path.is_file() for path in comparison_paths.values())
comparisons = {scope: json.loads(path.read_text(encoding='utf-8')) for scope, path in comparison_paths.items()}

In [3]:
metric_rows = []
distribution_rows = []
for scope, comparison in comparisons.items():
    for calibrated, metric_key, distribution_key in (
        (False, 'metric_absolute_differences', 'score_distributions'),
        (True, 'calibrated_metric_absolute_differences', 'calibrated_score_distributions'),
    ):
        for learner, metrics in comparison[metric_key].items():
            metric_rows.append({'scope': scope, 'calibrated': calibrated, 'learner': learner, **metrics})
        for learner, distribution in comparison[distribution_key].items():
            distribution_rows.append({
                'scope': scope,
                'calibrated': calibrated,
                'learner': learner,
                **distribution,
            })
display(pl.DataFrame(metric_rows))
display(pl.DataFrame(distribution_rows))
display({scope: report['best_params'] for scope, report in comparisons.items()})
display({scope: report['timing_seconds'] for scope, report in comparisons.items()})

scope,calibrated,learner,qini_auc,uplift_auc,uplift_at_10,uplift_at_20,uplift_at_50
str,bool,str,f64,f64,f64,f64,f64
"""product""",false,"""s""",0.001352,0.001545,0.0,0.0,0.002354
"""product""",false,"""t""",0.00465,0.00436,0.009409,0.033665,0.001841
"""product""",false,"""x""",0.000679,0.000614,0.002267,0.008808,0.000136
"""product""",true,"""s""",0.000966,0.00105,0.001196,0.000241,0.003816
"""product""",true,"""t""",0.002052,0.001729,0.006391,0.000039,0.00101
…,…,…,…,…,…,…,…
"""channel""",false,"""t""",0.002141,0.002593,0.004203,0.007755,0.003183
"""channel""",false,"""x""",0.000174,0.000075,0.004674,0.003583,0.000515
"""channel""",true,"""s""",0.001022,0.000991,0.004192,0.003156,0.005206


scope,calibrated,learner,reference_mean,fmlib_mean,reference_std,fmlib_std,max_quantile_difference
str,bool,str,f64,f64,f64,f64,f64
"""product""",false,"""s""",-0.0586,-0.058052,0.014075,0.013688,0.006341
"""product""",false,"""t""",-0.099397,-0.099313,0.044808,0.047381,0.008339
"""product""",false,"""x""",-0.142285,-0.142278,0.030601,0.030552,0.003065
"""product""",true,"""s""",-0.187396,-0.189639,0.09394,0.096454,0.015239
"""product""",true,"""t""",-0.186348,-0.189397,0.117299,0.123501,0.020023
…,…,…,…,…,…,…,…
"""channel""",false,"""t""",-0.099608,-0.10021,0.049071,0.045618,0.009354
"""channel""",false,"""x""",-0.142388,-0.142751,0.035592,0.034645,0.004365
"""channel""",true,"""s""",-0.187377,-0.190279,0.097897,0.098239,0.035283


{'product': {'reference': {"<class 'causalml.inference.meta.slearner.BaseSClassifier'>_0": {'iterations': 6,
    'depth': 2,
    'l2_leaf_reg': 1.0,
    'bagging_temperature': 0.25,
    'learning_rate': 0.1},
   "<class 'causalml.inference.meta.slearner.BaseSClassifier'>_1": {'iterations': 6,
    'depth': 2,
    'l2_leaf_reg': 1.0,
    'bagging_temperature': 0.25,
    'learning_rate': 0.1},
   "<class 'causalml.inference.meta.tlearner.BaseTClassifier'>_0": {'iterations': 6,
    'depth': 2,
    'l2_leaf_reg': 1.0,
    'bagging_temperature': 0.25,
    'learning_rate': 0.1},
   "<class 'causalml.inference.meta.tlearner.BaseTClassifier'>_1": {'iterations': 6,
    'depth': 2,
    'l2_leaf_reg': 1.0,
    'bagging_temperature': 0.25,
    'learning_rate': 0.1},
   "<class 'causalml.inference.meta.xlearner.BaseXClassifier'>_0": {'iterations': 6,
    'depth': 2,
    'l2_leaf_reg': 1.0,
    'bagging_temperature': 0.25,
    'learning_rate': 0.1},
   "<class 'causalml.inference.meta.xlearner.BaseXC

{'product': {'reference': {'train': 63.815012599981856,
   'evaluate': 257.3055802999879},
  'fmlib': {'train': 20.490816100034863, 'evaluate': 7.061598400003277}},
 'channel': {'reference': {'train': 57.45905799994944,
   'evaluate': 198.5082593000261},
  'fmlib': {'train': 49.75415739999153, 'evaluate': 18.140664599952288}}}